<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/CRIR003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------------------------------------------------------
# CRIT003 — λ-scan with gentle spectral window
# ---------------------------------------------------------

import os
import glob
import numpy as np
from scipy.fft import fft2, ifft2, fftshift, ifftshift

# ---------------------------------------------------------
# Gentle spectral window builder (same as CRIT002 gentle)
# ---------------------------------------------------------
def build_spectral_window(N, k0_lattice=0.25, width=0.06):
    ky = np.fft.fftfreq(N) * N
    kx = np.fft.fftfreq(N) * N
    KX, KY = np.meshgrid(kx, ky)
    k_mag = np.sqrt(KX**2 + KY**2) / (N/2)

    band = np.exp(-0.5 * ((k_mag - k0_lattice) / width)**2)
    band /= (band.max() + 1e-12)

    return fftshift(band)

# ---------------------------------------------------------
# Evolve step (BOA010 + gentle window)
# ---------------------------------------------------------
def evolve_step(field, lam, params, spectral_window, alpha):
    dt      = params['dt']
    kappa   = params['kappa']
    delta   = params['delta']
    epsilon = params['epsilon']
    gamma_0 = params['gamma_0']
    xi_std  = params['xi_std']

    A = field

    # Laplacian
    laplacian = (
        np.roll(A, 1, 0) + np.roll(A, -1, 0) +
        np.roll(A, 1, 1) + np.roll(A, -1, 1) - 4 * A
    )

    # Local nonlinear dynamics
    xi = np.random.normal(0, xi_std, A.shape)
    dA_local = (-kappa * A + delta * A**2 + epsilon * A**3 + gamma_0 * laplacian + xi)

    # Gentle spectral shaping
    A_fft = fftshift(fft2(A))
    A_fft_filtered = A_fft * (1.0 + alpha * lam * spectral_window)
    A_filtered = np.real(ifft2(ifftshift(A_fft_filtered)))

    # Combine
    dA = dA_local + alpha * lam * (A_filtered - A)

    A_new = A + dt * dA
    return np.nan_to_num(A_new)

# ---------------------------------------------------------
# Correlation-length estimator (k_peak, xi)
# ---------------------------------------------------------
def estimate_correlation_length(frame):
    A = frame - np.mean(frame)
    fft_A = fftshift(fft2(A))
    power = np.abs(fft_A)**2

    N = A.shape[0]
    y, x = np.indices((N, N))
    center = (N // 2, N // 2)
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)

    tbin = np.bincount(r.ravel(), power.ravel())
    nr = np.bincount(r.ravel())
    radial_ps = tbin / (nr + 1e-10)

    k_vals = np.arange(len(radial_ps)) / (N / 2)
    k_peak = k_vals[np.argmax(radial_ps[1:]) + 1]
    xi_est = 1.0 / (k_peak + 1e-10)

    return k_peak, xi_est

# ---------------------------------------------------------
# Global parameters
# ---------------------------------------------------------
N = 256
steps = 5000
save_every = 200

params = dict(
    dt=0.01,
    kappa=0.2,
    delta=0.005,
    epsilon=0.0005,
    gamma_0=0.25,
    xi_std=0.02
)

alpha = 0.02  # gentle window strength
spectral_window = build_spectral_window(N, k0_lattice=0.25, width=0.06)

# λ values to scan
lambda_values = [0.0, 0.5, 1.0, 1.5, 2.0]

results = []

# ---------------------------------------------------------
# λ-scan loop
# ---------------------------------------------------------
for lam in lambda_values:
    folder = f"/content/lambda_{lam}_scan/"
    os.makedirs(folder, exist_ok=True)

    # fresh initial condition per λ
    A = np.random.normal(0, 0.1, (N, N))

    print(f"\n=== Running λ = {lam} ===")
    for t in range(steps):
        A = evolve_step(A, lam, params, spectral_window, alpha)

        if t % save_every == 0:
            np.save(folder + f"frame_{t:05d}.npy", A)
            print(f"λ={lam}: saved frame at step {t}")

    # estimate correlation length from last frame
    k_peak, xi_est = estimate_correlation_length(A)
    results.append((lam, k_peak, xi_est))
    print(f"λ={lam}: k_peak={k_peak:.5f}, xi={xi_est:.5f}")

# ---------------------------------------------------------
# Print summary table
# ---------------------------------------------------------
print("\n=== λ-scan summary (gentle window) ===")
print("lambda\tk_peak\t\txi")
for lam, k_peak, xi_est in results:
    print(f"{lam:.2f}\t{k_peak:.5f}\t{xi_est:.5f}")



=== Running λ = 0.0 ===
λ=0.0: saved frame at step 0
λ=0.0: saved frame at step 200
λ=0.0: saved frame at step 400
λ=0.0: saved frame at step 600
λ=0.0: saved frame at step 800
λ=0.0: saved frame at step 1000
λ=0.0: saved frame at step 1200
λ=0.0: saved frame at step 1400
λ=0.0: saved frame at step 1600
λ=0.0: saved frame at step 1800
λ=0.0: saved frame at step 2000
λ=0.0: saved frame at step 2200
λ=0.0: saved frame at step 2400
λ=0.0: saved frame at step 2600
λ=0.0: saved frame at step 2800
λ=0.0: saved frame at step 3000
λ=0.0: saved frame at step 3200
λ=0.0: saved frame at step 3400
λ=0.0: saved frame at step 3600
λ=0.0: saved frame at step 3800
λ=0.0: saved frame at step 4000
λ=0.0: saved frame at step 4200
λ=0.0: saved frame at step 4400
λ=0.0: saved frame at step 4600
λ=0.0: saved frame at step 4800
λ=0.0: k_peak=0.00781, xi=128.00000

=== Running λ = 0.5 ===
λ=0.5: saved frame at step 0
λ=0.5: saved frame at step 200
λ=0.5: saved frame at step 400
λ=0.5: saved frame at step 600